# pandas: Series y DataFrame

## Introducción

pandas es una biblioteca para la manipulación y el análisis de datos. Se creó en 2008 como respuesta al uso creciente de Python en aplicaciones científicas tradicionalmente dominadas por **R**, MATLAB o SAS, apoyándose en la madurez y la estabilidad de **NumPy** y **SciPy**. Su nombre viene de ***Pan**el **Da**ta* (datos de panel), un término habitual en estadística y econometría para los datasets multidimensionales.

Permite:
- Importar fácilmente desde CSV, JSON, Excel, SQL, etc.
- Operaciones de manipulación: selección, filtrado, agregación.
- Limpieza de datos (también llamada *data cleansing*).
- *Data wrangling* o *data munging*: transformar datos entre formatos.

pandas tiene dos estructuras:
- **Series**: un array unidimensional con etiquetas
- **DataFrame**: una tabla bidimensional, con filas y columnas etiquetadas

Antes había una tercera, el `Panel` tridimensional, eliminada en pandas 0.25 (2019): un DataFrame con un índice jerárquico (un *MultiIndex*) hace el mismo trabajo.

Documentación oficial: https://pandas.pydata.org/docs/

In [ ]:
import pandas as pd

## Series

El tipo **Series** es un array unidimensional que contiene una secuencia de valores y una secuencia de etiquetas asociadas a los valores, llamada índice. La existencia de este índice explícito (que puede ser de cualquier tipo inmutable) es la diferencia principal con un vector de NumPy, que tiene un índice implícito (una secuencia de enteros que indica la posición). Los índices de una Series son como los de un diccionario, mientras que los de **NumPy** son como los de una lista.

### Estructura

In [ ]:
example_series = pd.Series([1, 2, 3, 4, 5, 6])  # Series con índice implícito, ya que parte de una lista
print(example_series)
print(type(example_series))

0    1
1    2
2    3
3    4
4    5
5    6
dtype: int64
<class 'pandas.Series'>


Igual que NumPy crea arrays a partir de listas, pandas crea una Series a partir de un diccionario: las claves se convierten en el índice, y los valores del diccionario en los valores de la Series.

In [ ]:
student_grades = pd.Series({'Student 1': 5, 'Student 2': 10, 'Student 3': 7, 'Student 4': 8})
student_grades

Student 1     5
Student 2    10
Student 3     7
Student 4     8
dtype: int64

In [ ]:
pd.Series([5, 10, 7, 8], index=["Student 1", "Student 2", "Student 3", "Student 4"])  # lo mismo, a partir de valores y etiquetas

Student 1     5
Student 2    10
Student 3     7
Student 4     8
dtype: int64

In [ ]:
occupied_seats = pd.Series({1: "Pepe Pérez", 7: "Juan Gómez", 6: "Ana López", 2: "María García", 5: "Luisa Martínez"})
occupied_seats

1        Pepe Pérez
7        Juan Gómez
6         Ana López
2      María García
5    Luisa Martínez
dtype: str

### Acceder a los elementos de una Series

Cuidado cuando las etiquetas del índice son números: una etiqueta y una posición pueden ser entonces el mismo número y apuntar a elementos distintos.

Para seleccionar por etiqueta, usa el atributo **loc**; para seleccionar por posición, usa **iloc** (de *integer location*). El operador de indexación **[]** por sí solo también selecciona por etiqueta, como con un diccionario.

In [ ]:
print(occupied_seats[7])  # el valor con etiqueta 7
print(occupied_seats.loc[7])  # lo mismo, explícitamente por etiqueta
print(occupied_seats.iloc[1])  # el valor en la posición 1

Juan Gómez
Juan Gómez
Juan Gómez


In [ ]:
print(student_grades["Student 1"])  # el valor con etiqueta "Student 1"
print(student_grades.loc["Student 1"])  # lo mismo, explícitamente por etiqueta
print(student_grades.iloc[0])  # el valor en la posición 0

5
5
5


Un entero dentro de **[]** es siempre una etiqueta, incluso cuando las etiquetas son texto: `student_grades[0]` busca una etiqueta `0`, que no existe. Antes de pandas 3.0 recurría a la posición 0, con un aviso; ahora da un `KeyError`:

In [ ]:
try:
    student_grades[0]
except KeyError as error:
    print(f"KeyError: {error}")

KeyError: 0


En cambio, un *slice* de enteros dentro de **[]** sigue significando posiciones, incluso cuando las etiquetas son enteros: `occupied_seats[0:2]` da los dos primeros asientos, no los asientos con etiquetas 0 y 1. Esa incoherencia es la razón por la que **loc** e **iloc** son la opción segura siempre que haya un número de por medio.

In [ ]:
occupied_seats[0:2]

1    Pepe Pérez
7    Juan Gómez
dtype: str

In [ ]:
# Modificar valores por etiquetas
student_grades['Student 1'] = 10
student_grades['Student 3':] = 5  # desde la etiqueta 'Student 3' hasta el final: un slice de etiquetas incluye los dos extremos
student_grades

Student 1    10
Student 2    10
Student 3     5
Student 4     5
dtype: int64

In [ ]:
print(student_grades.mean())  # media de las notas
print(student_grades.std())  # desviación típica

7.5
2.886751345948129


In [ ]:
print(student_grades.describe())  # estadísticos descriptivos de las notas

count     4.000000
mean      7.500000
std       2.886751
min       5.000000
25%       5.000000
50%       7.500000
75%      10.000000
max      10.000000
dtype: float64


## DataFrame

### Estructura de un DataFrame

Un **DataFrame** es una estructura de datos tabular bidimensional, con filas y columnas etiquetadas. Es parecida a una tabla de una base de datos relacional (SQL). Se puede considerar una colección de Series que comparten el mismo índice. Es la estructura de datos más usada de pandas.

In [ ]:
pd.DataFrame({'Grades': student_grades})  # un DataFrame a partir de la Series, dando nombre a su columna

,Grades
Student 1,10
Student 2,10
Student 3,5
Student 4,5


In [ ]:
# Un DataFrame con las notas de varios alumnos en varios módulos
pd.DataFrame({'PIA': student_grades, 'SAA': [5, 6, 7, 8], 'MIA': [9, 8, 7, 6], 'SBD': [10, 9, 8, 7], 'BDA': [6, 7, 8, 9]})

,PIA,SAA,MIA,SBD,BDA
Student 1,10,5,9,10,6
Student 2,10,6,8,9,7
Student 3,5,7,7,8,8
Student 4,5,8,6,7,9


En el caso anterior, el índice de `student_grades` se convirtió en el índice del DataFrame. La primera columna viene de una Series, y las siguientes de listas.

In [ ]:
# Otra opción es dar el índice explícitamente, con todas las demás notas como listas
pd.DataFrame({'PIA': student_grades, 'SAA': [5, 6, 7, 8], 'MIA': [9, 8, 7, 6], 'SBD': [10, 9, 8, 7], 'BDA': [6, 7, 8, 9]},
             index=['Wrong Name', 'Student 2', 'Student 3', 'Student 4'])

,PIA,SAA,MIA,SBD,BDA
Wrong Name,NaN,5,9,10,6
Student 2,10.0,6,8,9,7
Student 3,5.0,7,7,8,8
Student 4,5.0,8,6,7,9


Nos hemos equivocado en el nombre de un alumno, y como la primera lista de notas (PIA) era una Series, no encuentra la nota para el índice 'Wrong Name' y devuelve **NaN (Not a Number)** (una constante de NumPy). Para evitarlo, podemos crear un DataFrame a partir de un diccionario de listas en lugar de un diccionario de Series. Las demás notas son listas simples sin índice, así que se suponen correctas.
Aun así, en este tipo de procesos es importante estar alerta. Que las notas de cada alumno solo se identifiquen por su posición en una lista no es muy robusto, ya que si se añade un alumno o se cambia el orden de los alumnos, las notas se asignarán a alumnos distintos. Es mejor usar un diccionario de Series, ya que el índice explícito permite identificar correctamente a cada alumno.

La siguiente solución es más robusta:

In [ ]:
grades_pia = pd.Series({'Marvin Minsky': 5.7, 'John McCarthy': 6.5, 'Claude Shannon': 6.5, 'Alan Turing': 7.0})
grades_saa = pd.Series({'Marvin Minsky': 8.0, 'John McCarthy': 8.5, 'Claude Shannon': 8.0, 'Alan Turing': 9.0})
grades_mia = pd.Series({'Marvin Minsky': 7.0, 'John McCarthy': 6.0, 'Claude Shannon': 6.0, 'Alan Turing': 7.0})
grades_sbd = pd.Series({'Marvin Minsky': 9.0, 'John McCarthy': 9.0, 'Claude Shannon': 9.0, 'Alan Turing': 10.0})
grades_bda = pd.Series({'John McCarthy': 7.8, 'Claude Shannon': 6.9, 'Alan Turing': 9.9, 'Marvin Minsky': 10})  # otro orden: las Series se alinean por etiqueta

grades_df = pd.DataFrame({'PIA': grades_pia, 'SAA': grades_saa, 'MIA': grades_mia, 'SBD': grades_sbd, 'BDA': grades_bda})
grades_df

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


Un par de problemas a tener en cuenta al usar *strings* como índices en el análisis de datos:
- Puede que no sean únicos (dos personas pueden llamarse igual)
- Puede haber variaciones en cómo se escriben los nombres (por ejemplo, con mayúsculas o minúsculas) en distintas fuentes de datos.
Son dos de las razones por las que las bases de datos relacionales siempre usan claves primarias únicas indexadas, a menudo enteros autoincrementales que no tienen significado por sí mismos (claves subrogadas).

In [ ]:
grades_pia = pd.Series({'Marvin Minsky': 5.7, 'John McCarthy': 6.2, 'Claude Shannon': 6.5, 'Alan Turing': 7.0})
grades_saa = pd.Series({'marvin minsky': 8.0, 'McCarthy': 8.5, 'shannon': 8.0, 'Alan-Turing': 9.0})
grades_df_messy = pd.DataFrame({'PIA': grades_pia, 'SAA': grades_saa})
grades_df_messy

,PIA,SAA
Alan Turing,7.0,NaN
Alan-Turing,NaN,9.0
Claude Shannon,6.5,NaN
John McCarthy,6.2,NaN
Marvin Minsky,5.7,NaN
McCarthy,NaN,8.5
marvin minsky,NaN,8.0
shannon,NaN,8.0


### Información sobre un DataFrame

In [ ]:
grades_df.info()  # información sobre el DataFrame

<class 'pandas.DataFrame'>
Index: 4 entries, Alan Turing to Marvin Minsky
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PIA     4 non-null      float64
 1   SAA     4 non-null      float64
 2   MIA     4 non-null      float64
 3   SBD     4 non-null      float64
 4   BDA     4 non-null      float64
dtypes: float64(5)
memory usage: 244.0 bytes


In [ ]:
grades_df.head()  # Primeras 5 filas (en este caso solo hay 4; normalmente trabajaremos con datasets mucho más grandes y será útil ver solo las primeras filas para hacernos una idea de los datos)

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


In [ ]:
grades_df.shape  # número de filas y columnas

(4, 5)

In [ ]:
grades_df.keys()  # objeto "Index" con los nombres de las columnas y su tipo

Index(['PIA', 'SAA', 'MIA', 'SBD', 'BDA'], dtype='str')

In [ ]:
grades_df.columns  # Equivale a lo anterior, pero solo para DataFrame (el método keys() también sirve para obtener las claves de una Series)

Index(['PIA', 'SAA', 'MIA', 'SBD', 'BDA'], dtype='str')

In [ ]:
grades_df.dtypes  # tipos de datos de las columnas

PIA    float64
SAA    float64
MIA    float64
SBD    float64
BDA    float64
dtype: object

In [ ]:
grades_df.index  # etiquetas de las filas

Index(['Alan Turing', 'Claude Shannon', 'John McCarthy', 'Marvin Minsky'], dtype='str')

### Escribir y leer archivos de datos

Pandas ofrece una gran variedad de funciones para importar y exportar datos desde y hacia archivos. Sin entrar en profundidad, como ejemplo podemos guardar el DataFrame `grades_df` en un **archivo CSV** con la función **to_csv** y recuperarlo con la función **read_csv**.

In [ ]:
grades_df.to_csv('data/grades.csv')  # el directorio data debe existir
df = pd.read_csv('data/grades.csv', index_col=0)
df

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


El parámetro `index_col=0` indica que la primera columna del archivo csv es el índice explícito del DataFrame; si no se especifica, se crea un índice implícito.

In [ ]:
pd.read_csv('data/grades.csv')

,Unnamed: 0,PIA,SAA,MIA,SBD,BDA
0,Alan Turing,7.0,9.0,7.0,10.0,9.9
1,Claude Shannon,6.5,8.0,6.0,9.0,6.9
2,John McCarthy,6.5,8.5,6.0,9.0,7.8
3,Marvin Minsky,5.7,8.0,7.0,9.0,10.0
